# Practice: one model, three solvers

Notes at <https://dse.iskh.me/cons-practice>.

## The model object

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import lognorm
from scipy.optimize import minimize_scalar, brentq

class deaton():
    '''Stochastic consumption-savings model of Deaton (1991) with log-normal income'''

    def __init__(self, beta=0.9, R=1.05, sigma=0.5, Mbar=10, ngrid=100, nchgrid=250, nquad=10):
        self.beta, self.R, self.sigma = beta, R, sigma   # discount factor, gross return, income std
        self.Mbar = Mbar                                  # upper bound of the state grid
        self.ngrid, self.nchgrid, self.nquad = ngrid, nchgrid, nquad
        self.eps = np.finfo(float).eps                    # smallest positive float
        self.grid = np.linspace(self.eps, Mbar, ngrid)    # state grid on (0, Mbar]
        self.share = np.linspace(self.eps, 1, nchgrid)    # choice grid as shares of wealth
        x, w = np.polynomial.legendre.leggauss(nquad)     # Gauss-Legendre nodes and weights on [-1,1]
        self.quadp = lognorm.ppf((x + 1) / 2, sigma)      # income nodes y_k = F^{-1}(q_k)
        self.quadw = w / 2                                # weights, sum to one

    def u(self, c):
        return np.log(c)                                  # utility

    def next_wealth(self, M, c, y):
        return self.R * (M - c) + y                       # budget constraint

def interp(x, xp, fp):
    '''Linear interpolation, extrapolated linearly beyond the last grid point'''
    slope = (fp[-1] - fp[-2]) / (xp[-1] - xp[-2])
    return np.where(x > xp[-1], fp[-1] + slope * (x - xp[-1]), np.interp(x, xp, fp))

## Computing expectations

In [ ]:
def quadrature_hermite(sigma, n):
    '''Gauss-Hermite nodes and weights for expectations of functions of a log-normal income'''
    x, w = np.polynomial.hermite.hermgauss(n)          # weight exp(-x^2) on the real line
    return np.exp(np.sqrt(2) * sigma * x), w / np.sqrt(np.pi)   # nodes y_k, weights sum to one

def quadrature_cdf(sigma, n):
    '''Gauss-Legendre on the CDF: nodes y_k = F^{-1}(q_k)'''
    x, w = np.polynomial.legendre.leggauss(n)          # Gauss-Legendre on [-1,1]
    return lognorm.ppf((x + 1) / 2, sigma), w / 2      # rescaled to [0,1], weights sum to one

sigma = 0.5
exact = np.exp(sigma**2 / 2)                           # E(y) = E(1/y) = exp(sigma^2/2)
print(f'{"nodes":>7} {"Gauss-Hermite":>14} {"Legendre on CDF":>16}   errors in E(1/y)')
for n in (3, 5, 10, 20):
    errs = [abs(w @ (1 / y) - exact) for y, w in (quadrature_hermite(sigma, n), quadrature_cdf(sigma, n))]
    print(f'{n:>7} {errs[0]:>14.1e} {errs[1]:>16.1e}')
rng = np.random.default_rng(2026)
print(f'{"draws":>7} {"Monte Carlo":>14} {"standard error":>16}')
for N in (100, 1_000, 10_000, 100_000):
    g = 1 / lognorm.rvs(sigma, size=N, random_state=rng)
    print(f'{N:>7} {abs(g.mean() - exact):>14.1e} {g.std() / np.sqrt(N):>16.1e}')

## VFI with discretized choice

In [ ]:
def bellman_discretized(m, V0):
    '''Bellman operator with discretized choice, V0 is a vector of values on the state grid'''
    c = m.share[:, None, None] * m.grid[None, :, None]   # choices x states x 1, scaled to [0, M]
    M1 = m.next_wealth(m.grid[None, :, None], c, m.quadp[None, None, :])   # choices x states x nodes
    EV = interp(M1, m.grid, V0) @ m.quadw                 # expected value, choices x states
    X = m.u(c[:, :, 0]) + m.beta * EV                     # maximand of the Bellman equation
    i = np.argmax(X, axis=0)                              # optimal choice in every state
    j = np.arange(m.ngrid)
    return X[i, j], c[i, j, 0]

def solve_vfi(m, bellman, maxiter=500, tol=1e-4, callback=None):
    '''Value function iterations with the given Bellman operator, returns the policy'''
    V0 = m.u(m.grid)                                      # start from consuming everything
    for i in range(maxiter):
        V1, c1 = bellman(m, V0)
        err = np.max(np.abs(V1 - V0))
        if callback: callback(iter=i, value=V1, policy=c1, err=err)
        if err < tol: break
        V0 = V1
    else:
        raise RuntimeError('No convergence: maximum number of iterations achieved')
    return c1

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
for sigma in (0.0001, 0.5, 0.75):
    ms = deaton(sigma=sigma)
    last = {}                                             # the callback keeps the last iteration
    c = solve_vfi(ms, bellman_discretized, callback=lambda **kw: last.update(kw))
    ax1.plot(ms.grid[1:], last['value'][1:], label=f'σ = {sigma}')
    ax2.plot(ms.grid, c, label=f'σ = {sigma}')
ax2.plot(ms.grid, ms.grid, 'k:', lw=0.8, label='c = M')
ax2.set(ylim=(0, 3))
for ax, t in ((ax1, 'Value function'), (ax2, 'Policy function')):
    ax.set(xlabel='Wealth, M', title=t)
    ax.grid(color='0.9')
    ax.legend()
fig.suptitle('VFI with discretized choice, 100 state and 250 choice points')
plt.show()

## VFI with continuous choice

In [ ]:
def bellman_continuous(m, V0):
    '''Bellman operator with continuous choice, solved by Brent's method in every state'''
    def minus_maximand(c, M):
        EV = interp(m.next_wealth(M, c, m.quadp), m.grid, V0) @ m.quadw
        return -(m.u(c) + m.beta * EV)                    # minus: minimize_scalar minimizes
    V1, c1 = np.empty(m.ngrid), np.empty(m.ngrid)
    for j, M in enumerate(m.grid):
        if M <= m.grid[0]:
            c1[j] = M                                     # no room to optimize at the first point
        else:
            res = minimize_scalar(minus_maximand, args=(M,), method='bounded',
                                  bounds=(m.eps, M), options={'xatol': 1e-10})
            c1[j] = res.x
        V1[j] = -minus_maximand(c1[j], M)
    return V1, c1

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
for sigma in (0.0001, 0.5, 0.75):
    ms = deaton(sigma=sigma)
    last = {}                                             # the callback keeps the last iteration
    c = solve_vfi(ms, bellman_continuous, callback=lambda **kw: last.update(kw))
    ax1.plot(ms.grid[1:], last['value'][1:], label=f'σ = {sigma}')
    ax2.plot(ms.grid, c, label=f'σ = {sigma}')
ax2.plot(ms.grid, ms.grid, 'k:', lw=0.8, label='c = M')
ax2.set(ylim=(0, 3))
for ax, t in ((ax1, 'Value function'), (ax2, 'Policy function')):
    ax.set(xlabel='Wealth, M', title=t)
    ax.grid(color='0.9')
    ax.legend()
fig.suptitle('VFI with truly continuous choice, 100 state and 250 choice points')
plt.show()

## Time iterations on the Euler equation

In [ ]:
def euler_residual(m, c, M, c0):
    '''LHS - RHS of the Euler equation at (M, c), with next period policy c0 on the grid'''
    M1 = m.next_wealth(np.atleast_1d(M)[:, None], np.atleast_1d(c)[:, None], m.quadp[None, :])
    c1 = np.maximum(interp(M1, m.grid, c0), m.eps)        # next period consumption, kept positive
    return 1 / c - m.beta * m.R * ((1 / c1) @ m.quadw).squeeze()   # marginal utility u'(c) = 1/c

def solve_timeiter(m, maxiter=500, tol=1e-4, callback=None):
    '''Time iterations: solve the Euler equation for c in every state, returns the policy'''
    c0 = m.grid.copy()                                    # start from consuming everything
    for i in range(maxiter):
        c1 = np.empty(m.ngrid)
        for j, M in enumerate(m.grid):
            if M <= m.grid[0] or euler_residual(m, M, M, c0) > 0:
                c1[j] = M                                 # constraint binds: consume everything
            else:
                c1[j] = brentq(lambda c: euler_residual(m, c, M, c0), m.eps, M, xtol=1e-12)
        err = np.max(np.abs(c1 - c0))
        if callback: callback(iter=i, policy=c1, err=err)
        if err < tol: break
        c0 = c1
    else:
        raise RuntimeError('No convergence: maximum number of iterations achieved')
    return c1

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
for sigma in (0.0001, 0.5, 0.75):
    ms = deaton(sigma=sigma)
    last = {}                                             # the callback keeps the last iteration
    c = solve_timeiter(ms, callback=lambda **kw: last.update(kw))
    ax2.plot(ms.grid, c, label=f'σ = {sigma}')
ax2.plot(ms.grid, ms.grid, 'k:', lw=0.8, label='c = M')
ax2.set(ylim=(0, 3))
for ax, t in ((ax1, 'Value function'), (ax2, 'Policy function')):
    ax.set(xlabel='Wealth, M', title=t)
    ax.grid(color='0.9')
ax2.legend()
fig.suptitle('Time iterations, 100 state and 250 choice points')
plt.show()

## Simulation and Keane's test

In [ ]:
def simulate(m, c, M0, T=50, seed=None):
    '''Simulate T periods from initial wealth M0 (scalar or array) with the policy c on the grid'''
    rng = np.random.default_rng(seed)
    M0 = np.atleast_1d(M0).astype(float)
    M, C = np.empty((M0.size, T)), np.empty((M0.size, T))
    M[:, 0] = M0
    for t in range(T):
        C[:, t] = np.minimum(interp(M[:, t], m.grid, c), M[:, t])   # no borrowing
        if t < T - 1:
            y = lognorm.rvs(m.sigma, size=M0.size, random_state=rng)
            M[:, t + 1] = m.next_wealth(M[:, t], C[:, t], y)
    return M, C

In [ ]:
k = deaton(beta=0.9, R=1/0.9, sigma=1e-10, nquad=2)       # deterministic income y = 1, beta*R = 1
M0, T = 5.0, 100
cstar = ((k.R - 1) * M0 + 1) / k.R
fig, ax = plt.subplots(figsize=(10, 4))
for name, solver in (('VFI, discretized choice', lambda: solve_vfi(k, bellman_discretized, tol=1e-8, maxiter=2000)),
                     ('VFI, continuous choice', lambda: solve_vfi(k, bellman_continuous, tol=1e-8, maxiter=2000)),
                     ('time iterations', lambda: solve_timeiter(k, tol=1e-10, maxiter=2000))):
    _, C = simulate(k, solver(), M0, T)
    ax.plot(C[0], label=name)
    print(f'{name:<25} max |c_t / c* - 1| = {np.max(np.abs(C[0] / cstar - 1)):.1e}')
ax.axhline(cstar, c='k', lw=0.8, ls=':', label='c*')
ax.set(xlabel='period', ylabel='simulated consumption', title="Keane's test")
ax.grid(color='0.9'); ax.legend()
plt.show()

## Speed and accuracy compared

In [ ]:
def accuracy(m, c, factor=10, rtol=1e-6):
    '''Mean absolute relative Euler residual of the policy c on a dense grid, interior points only'''
    M = np.linspace(m.grid[1], m.Mbar, m.ngrid * factor)  # dense grid, first point skipped
    cM = interp(M, m.grid, c)                             # interpolated policy
    er = euler_residual(m, cM, M, c) * cM                 # relative Euler residual, divided by u'(c) = 1/c
    interior = cM < M * (1 - rtol)                        # drop the corner solutions c = M
    return np.mean(np.abs(er[interior]))

In [ ]:
solvers = {'VFI, discretized choice': lambda m: solve_vfi(m, bellman_discretized, tol=1e-8, maxiter=2000),
           'VFI, continuous choice': lambda m: solve_vfi(m, bellman_continuous, tol=1e-8, maxiter=2000),
           'time iterations': lambda m: solve_timeiter(m, tol=1e-8, maxiter=2000)}
sigmas, grids = (0.0001, 0.5, 0.75), (50, 100, 200, 400)
M0, T = 5.0, 100                                          # Keane's test: initial wealth and periods
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
for row, sigma in zip(axes, sigmas):
    keane_test = sigma < 0.01                             # sigma near zero: Keane's test with beta*R = 1
    for name, solve in solvers.items():
        keane, euler, runtime = [], [], []
        for n in grids:
            if keane_test:
                m = deaton(sigma=sigma, R=1/0.9, nquad=2, ngrid=n)   # beta = 0.9, so beta*R = 1
            else:
                m = deaton(sigma=sigma, ngrid=n)
            t0 = time.perf_counter()
            c = solve(m)
            runtime.append(time.perf_counter() - t0)
            euler.append(accuracy(m, c))
            if keane_test:
                _, C = simulate(m, c, M0, T, seed=2026)
                cstar = ((m.R - 1) * M0 + 1) / m.R
                keane.append(np.max(np.abs(C[0] / cstar - 1)))
        if keane_test:
            row[0].plot(grids, keane, 'o-', label=name)
        row[1].plot(grids, euler, 'o-', label=name)
        row[2].plot(grids, runtime, 'o-', label=name)
    if not keane_test:
        row[0].axis('off')
        row[0].text(0.5, 0.5, f"σ = {sigma}\nKeane's test needs σ ≈ 0 and βR = 1",
                    ha='center', va='center', transform=row[0].transAxes)
    titles = ("Keane's test, max |c_t / c* - 1|", 'Mean |Euler residual|', 'Run time, s')
    for ax, t in zip(row, titles):
        ax.set(xscale='log', yscale='log', xlabel='State grid points',
               title=f'{t}, σ = {sigma}' + (', βR = 1' if keane_test else ''))
        ax.set_xticks(grids, [str(n) for n in grids])
        ax.grid(color='0.9')
axes[0, 0].legend()
fig.tight_layout()
plt.show()